# ずんだもん解説動画（Colab GPU版）

目的: 文字起こしはGemini（gemini-3.5-transcribe、GPUなし）、音声合成と動画の処理はColabのGPUで行う。

**準備（ご自身で行う）**
1. Colabで「ランタイム → ランタイムのタイプを変更 → GPU（T4など）」を選ぶ。
2. 左のキー（🔑 シークレット）に `GEMINI_API_KEY` を登録し、「ノートブックからのアクセス」をオンにする。キーの値は、ノートブックやチャットに書かない。
3. 次の2つのZIPを手元に用意する（私が作成済み）。
   - `voicebox-code.zip`（処理のプログラム）
   - `project-assets.zip`（立ち絵・フォント・背景・研究の依頼ファイル）

**手順**: 上から順に、1つずつ実行する。途中で止まったら、その時のエラーの行を教えてください。

**注意**: GPU版のVOICEVOXは1エンジンで動く（`WORKERS = 1`）。文字起こしはGeminiを使うため、料金はAPIアカウントにかかる。

In [ ]:
# 1. GPUを確認する（Tesla T4 などが表示されればOK）
!nvidia-smi

In [ ]:
# 2. 必要な道具を入れる（数分かかる）
!apt-get -qq update
!apt-get -qq install -y p7zip-full fonts-noto-cjk > /dev/null
!pip install -q moviepy==2.2.1 CairoSVG==2.9.1 playwright==1.64.0 budoux==0.9.0 pysubs2==1.8.1 "Pillow>=10.4,<12" "google-auth>=2.35,<3"
!playwright install --with-deps chromium > /dev/null

In [ ]:
# 3. ZIPを2つアップロードする（voicebox-code.zip と project-assets.zip を選ぶ）
from google.colab import files
uploaded = files.upload()
print(sorted(uploaded))

In [ ]:
# 4. ZIPを展開する
!rm -rf /content/voicebox /content/project
!unzip -q -o voicebox-code.zip -d /content/voicebox
!unzip -q -o project-assets.zip -d /content/project
!ls /content/voicebox | head -30
!ls /content/project

In [ ]:
# 5. Gemini のキーを読み込む（キーの値は表示しない）
import os
from google.colab import userdata
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
os.environ['VOICEBOX_GEMINI_ROUTE'] = 'api_key'
print('GEMINI key loaded:', bool(os.environ['GEMINI_API_KEY']))

In [ ]:
# 6. GPU版のVOICEVOX（公式 Linux NVIDIA 0.25.2）を入れる。両方の分割ファイルの検証値を確かめる
!python /content/voicebox/tools/install_voicevox_gpu.py --directory /content/voicevox-nvidia-0.25.2
!ls /content/voicevox-nvidia-0.25.2 | head -20

In [ ]:
# 7. 素材と場所を登録する（初回だけ）。GPU版の指定は環境変数で行う
%cd /content/voicebox
!VOICEBOX_DEVICE=cuda python pipeline.py setup --engine-dir /content/voicevox-nvidia-0.25.2 --shell "/content/project/assets/characters/Shikokumetan_Zundamon_Shell(20230806)" --font /content/project/assets/fonts/ZenMaruGothic-Medium.ttf --background /content/project/assets/background/blackboard-user.jpg

In [ ]:
# 8. 研究と台本を作る（Gemini。資料の撮影を含む）
%cd /content/voicebox
!python tools/prepare_project.py --request /content/project/request/research-request.json --output /content/work/prepared-colab-1
!cat /content/work/prepared-colab-1/preparation.json | head -60

In [ ]:
# 9. 動画を作る（音声合成はGPU版VOICEVOX、文字起こしはGemini）
%cd /content/voicebox
!VOICEBOX_DEVICE=cuda python pipeline.py _run /content/work/prepared-colab-1/project.json --out /content/work/video-colab-1

In [ ]:
# 10. 結果を確かめる（使った場所とGPUの記録）
import json, subprocess
state = json.load(open('/content/work/video-colab-1/state.json', encoding='utf-8'))
print({k: state.get(k) for k in ('status', 'device', 'workers', 'duration', 'total_seconds', 'gpu_used', 'max_parallel_synthesis')})
print(subprocess.run(['ffprobe','-v','error','-show_entries','format=duration:stream=codec_type,width,height,r_frame_rate','-of','compact','/content/work/video-colab-1/video.mp4'], capture_output=True, text=True).stdout)

In [ ]:
# 11. 動画をパソコンへダウンロードする
from google.colab import files
files.download('/content/work/video-colab-1/video.mp4')
files.download('/content/work/video-colab-1/state.json')